In [ ]:
import os
import sys
## adding the root directory to the library directory so still can read utility functions
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root) # setting work directory

import matplotlib.pyplot as plt
import warnings
import pickle 
import pandas as pd
import numpy as np

from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
import lightgbm as lgb


def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    
    warnings.simplefilter("ignore")
    fxn()

In [ ]:
### reading in CHS data
ukb_raw = pd.read_csv("Data/final_datasets/UKB_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", sep = "\t", 
                      compression = "gzip")

ukb_raw.head(5)

In [ ]:
## there are couple columns we need to rename in UKB
ukb_raw = ukb_raw.rename(columns={'ERVV-1': 'ERVV.1', 'HLA-A': 'HLA.A', 'HLA-DRA': 'HLA.DRA', 'HLA-E':'HLA.E'})

In [ ]:
### read in MESA soma for the column names
MESA_soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz", 
                        sep = "\t", compression = "gzip").drop(columns={"bmi"})
MESA_soma["LEP"].describe()
MESA_olink = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", 
                         sep = "\t", compression = "gzip")

rank_by_gain = pd.read_csv("")

# Find subjects who are in both datasets

overlap_IDs = np.unique(list(set(np.array(MESA_olink["id"])).intersection(set(np.array(MESA_soma["id"])))))

MESA_olink  = MESA_olink[MESA_olink["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])
MESA_soma   = MESA_soma[MESA_soma["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])

In [ ]:
# common_cols = ["id"] + [col for col in MESA_soma.columns if col in MESA_olink.columns and col != "id"]
# MESA_soma  = MESA_soma[common_cols]
# MESA_olink = MESA_olink[common_cols]

In [ ]:
# common columns of the two dataset
common_cols = [col for col in ukb_raw.columns if col in MESA_olink.columns]

print(len(common_cols))

ukb_short = ukb_raw[common_cols]

In [ ]:
MESA_soma = MESA_soma.set_index("id")
MESA_olink = MESA_olink.set_index("id")

In [ ]:
MESA_soma['LEP'].describe()

In [ ]:
ukb_short['LEP'].describe()
ukb_short = ukb_short.set_index("id")

In [ ]:
ukb_olink = ukb_short

In [ ]:
protlist = MESA_soma.columns

print(protlist[1:5])
print(len(protlist))

In [ ]:
ukb_olink.shape

## Evaluating Protein performance

In [ ]:
%%time
results = []


from scipy.stats import pearsonr
olink_cols = set(MESA_olink.columns)   # what exists in olink

train_index = pickle.load(open("Data/helper_data/train_index.pkl", "rb"))
test_index = pickle.load(open("Data/helper_data/test_index.pkl", "rb"))

for protein_name in protlist:
    model_path = f"Data/soma_trained_models/olink_soma/3kolink_5k_log2scaled_soma/{protein_name}_hgb.pkl"
    reg = pickle.load(open(model_path, "rb"))
    print(f"load the model for {protein_name}")
    

    if protein_name in olink_cols:
        
        ## loading test set, predicted value and actual value
        y = MESA_soma[protein_name]
        
        X_protein = MESA_olink[protein_name]
        X_train_protein = X_protein.loc[train_index]
        X_test_protein = X_protein.loc[test_index]
        
        X_train = MESA_olink.loc[train_index]
        y_train = y.loc[train_index]
        
        X_test = MESA_olink.loc[test_index]
        y_test = y.loc[test_index]
        y_pred = reg.predict(X_test)
        
        # correlation between SOMA and OLINK full column
        pretrain_r, _ = pearsonr(y_train, X_train_protein)
        pretrain_r2 = pretrain_r **2
        
        # pred and observed comparison
        model_r, _ = pearsonr(y_test, y_pred)
        model_r2 = model_r**2
        
        ## post training correlation?
        posttrain_r,_ = pearsonr(y_pred, X_test_protein)
        posttrain_r2 = posttrain_r ** 2
        
        testset_r, _ = pearsonr(y_test, X_test_protein)
        testset_r2 = testset_r ** 2
        
        results.append({
            "protein": protein_name,
            "has_soma": protein_name in olink_cols,
            "model_r": model_r,
            "model_r2": model_r2,
            "pretrain_r": pretrain_r,
            "pretrain_rsq": pretrain_r2,
            "posttrain_r": posttrain_r,
            "posttrain_rsq": posttrain_r2,
            "testset_r": testset_r,
            "testset_r2": testset_r2
        })
    
    
    else:
        ## we can still calculate r2 for non overlapping ones
        y = MESA_soma[protein_name]
        X_test = MESA_olink.loc[test_index]
        y_test = y.loc[test_index]
        y_pred = reg.predict(X_test)
        

        # pred and observed comparison
        model_r, _ = pearsonr(y_test, y_pred)
        model_r2 = model_r**2
        
        results.append({
            "protein": protein_name,
            "has_soma": protein_name in olink_cols,
            "model_r": model_r,
            "model_r2": model_r2,
            "pretrain_r": np.nan,
            "pretrain_rsq": np.nan,
            "posttrain_r": np.nan,
            "posttrain_rsq": np.nan,
            "testset_r": np.nan,
            "testset_r2": np.nan
        })
    

In [ ]:
print(results)

In [ ]:
results_df = pd.DataFrame(results)
rsq_highest_row = results_df.loc[results_df["model_r2"].idxmax()]

## mean of rsq for all the proteins
print(f'Mean R square among proteins is: {results_df["model_r2"].mean():.4f}')
print(f'Max R square among proteins is: {results_df["model_r2"].max():.4f} for protein: {rsq_highest_row["protein"]}')

# correlation is between the soma measurement and olink measurement
print(f'Mean correlation among proteins is: {results_df["pretrain_rsq"].mean():.4f}')
print(f'Max correlation: {results_df["pretrain_rsq"].max():.4f} for protein: {rsq_highest_row["protein"]}')

In [ ]:
results_df.loc[results_df["protein"] == "LEP"]

In [ ]:
### saving results

results_df.to_csv("Results/metrics/HGB_3kolink_log2scaled_5ksoma_results.tsv", sep = "\t", index = False)

In [ ]:
ukb_olink = ukb_olink[MESA_olink.columns]

## Predicting UKB SOMA

In [ ]:
%%time

# ## validation section
# from scipy.stats import pearsonr

results = []

for protein_name in protlist:

    model_path = f"Data/soma_trained_models/olink_soma/3kolink_5k_log2scaled_soma/{protein_name}_hgb.pkl"
    reg = pickle.load(open(model_path, "rb"))

    print(f"load the model for {protein_name}")

    y_pred = reg.predict(ukb_olink)

    # temporary result table
    result = pd.DataFrame({
        "id": ukb_olink.index,
        protein_name: y_pred
    })

    # appending result
    results.append(result)

In [ ]:
## indexing
indexed = [df.set_index("id") for df in results]

ukb_soma_results = pd.concat(indexed, axis = 1, join = "inner").reset_index()

In [ ]:
ukb_soma_results.head(10)
ukb_soma_results.isna().sum().sum()

In [ ]:
%%time
ukb_soma_results.to_csv("Results/UKB_pred/HGB_predicted_UKB_3kolink_log2scaled_5ksoma.tsv", sep = "\t", index = False)